In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import importlib
import dataset
from image_builder import *
from data_prepare import *
import data_fetch
from data_fetch import *
from data_fetch import _download_single, _read_mops_csv

importlib.reload(dataset)
importlib.reload(data_fetch)

<module 'data_fetch' from '/home/iof_314707035/Jasper/Reimage_price_trends/data_fetch.py'>

In [2]:
from dataset import *
import os

START = "1993-01-01"
END = "1993-12-31"
TRAIN_END = "1993-10-31"
SAVE_DIR = "data"
Market = "US_1993_2020"
I = 5
R = 5
CRSP_DATA_DIR = resolve_crsp_data_dir()
print("CRSP data dir:", CRSP_DATA_DIR)

TICKERS, crsp_info = get_crsp_permno_universe(
    crsp_data_dir=CRSP_DATA_DIR,
    start=START,
    end=END,
    return_info=True,
)

print("permno 數量:", len(TICKERS))
display(crsp_info.head())

CRSP data dir: us_stock
permno 數量: 8081


,ticker,permno,first_year,last_year,rows
0,10001,10001,1993,1993,179
1,10002,10002,1993,1993,85
2,10003,10003,1993,1993,237
3,10009,10009,1993,1993,188
4,10010,10010,1993,1993,253


In [3]:
X_trainval, y_trainval, X_test, y_test, meta = build_research_dataset(
    tickers=TICKERS,
    start=START,
    end=END,
    I=I,
    R=R,
    train_end=TRAIN_END,
    save_dir=SAVE_DIR,
    Market=Market,
    price_source="crsp",
    crsp_data_dir=CRSP_DATA_DIR,
    checkpoint_every=1,
    resume=False,
    process_by="year",
    checkpoint_save_arrays=False,
    year_ticker_chunk_size=1000,
    max_workers=8,
)



Year-based CRSP build: pending years=[1993]

[I5/R5] build year: 1993


year 1993 chunks: 100%|██████████| 9/9 [03:20<00:00, 22.24s/it]


  done year 1993: 326062 images
  Checkpoint saved: 1 tickers done, 326062 images
  Checkpoint saved: 1 tickers done, 326062 images

總樣本數: 326062
標籤分布: up=0.491, down=0.509
  Train+Val: 271895  (up=0.493)
  Test     : 54167  (up=0.480)

撌脣摮 data/training_data/US_1993_2020/I5R5S5_week/


In [5]:
df = pd.read_parquet("us_stock/crsp_1993.parquet")
df.head()

,permno,date,openprc,askhi,bidlo,prc,vol,ret,cfacpr,cfacshr
0,10001,1993-01-04,14.5,14.5,14.5,14.5,150.0,0.035714,3.0,3.0
1,10003,1993-01-04,3.875,4.25,3.75,3.875,46376.0,0.0,1.0,1.0
2,10009,1993-01-04,12.25,12.25,11.75,12.25,1124.0,0.0,2.0,2.0
3,10010,1993-01-04,12.125,12.125,11.5,11.5,44685.0,-0.031579,1.35762,1.35762
4,10011,1993-01-04,4.625,4.8125,4.5,4.8125,45396.0,0.040541,1.0,1.0


In [17]:
df = df[df['permno'] == 10001]
df.head(30)

,permno,date,openprc,askhi,bidlo,prc,vol,ret,cfacpr,cfacshr
0,10001,1993-01-04,14.5,14.5,14.5,14.5,150.0,0.035714,3.0,3.0
18389,10001,1993-01-07,14.5,14.5,14.5,14.5,228.0,0.017544,3.0,3.0
24518,10001,1993-01-08,14.25,14.5,14.25,14.5,1375.0,0.0,3.0,3.0
30650,10001,1993-01-11,14.5,14.5,14.5,14.5,200.0,0.0,3.0,3.0
42937,10001,1993-01-13,14.0,14.0,13.75,13.75,700.0,-0.035088,3.0,3.0
49046,10001,1993-01-14,13.75,13.75,13.75,13.75,400.0,0.0,3.0,3.0
67472,10001,1993-01-19,13.75,13.75,13.75,13.75,200.0,-0.017857,3.0,3.0
73669,10001,1993-01-20,14.25,14.25,14.25,14.25,355.0,0.036364,3.0,3.0
79845,10001,1993-01-21,14.25,14.25,14.25,14.25,300.0,0.0,3.0,3.0
86016,10001,1993-01-22,13.75,13.75,13.75,13.75,200.0,-0.035088,3.0,3.0


In [4]:
I = 20
R = 20
Market = "US_1993_2020"

dataset_dir = f"data/training_data/{Market}/I{I}R{R}S{R}_period_end_month"

X_trainval = np.load(f"{dataset_dir}/X_trainval.npy")
y_trainval = np.load(f"{dataset_dir}/y_trainval.npy")

X_test = np.load(f"{dataset_dir}/X_test.npy")
y_test = np.load(f"{dataset_dir}/y_test.npy")

meta = pd.read_csv(f"{dataset_dir}/meta.csv")
meta_trainval = pd.read_csv(f"{dataset_dir}/meta_trainval.csv")
meta_test = pd.read_csv(f"{dataset_dir}/meta_test.csv")

print("X_trainval:", X_trainval.shape)
print("y_trainval:", y_trainval.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("meta_trainval:", meta_trainval.shape)
print("meta_test:", meta_test.shape)

display(meta.head())



X_trainval: (727713, 64, 60)
y_trainval: (727713,)
X_test: (1413250, 64, 60)
y_test: (1413250,)
meta_trainval: (727713, 13)
meta_test: (1413250, 13)


,ticker,start_date,date,label_end_date,label,ret,I,R,sample_step,sample_mode,sample_freq,price_source,ending_date
0,10001,1993-01-07,1993-02-26,1993-04-15,0,-0.008783,20,20,20,period_end,month,crsp,1993-02-26
1,10001,1993-02-12,1993-03-31,1993-05-14,1,0.111863,20,20,20,period_end,month,crsp,1993-03-31
2,10001,1993-05-26,1993-06-30,1993-08-05,1,0.006875,20,20,20,period_end,month,crsp,1993-06-30
3,10001,1993-07-22,1993-08-31,1993-10-06,1,0.081145,20,20,20,period_end,month,crsp,1993-08-31
4,10001,1993-08-25,1993-09-30,1993-11-03,1,0.012845,20,20,20,period_end,month,crsp,1993-09-30


In [5]:
import model
importlib.reload(model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [12]:
RUN_SEEDS = list(range(5))
ENSEMBLE_DIR = f"models/{Market}/I{I}R{R}"
EPOCHS = 50
BATCH_SIZE = 128
VAL_RATIO = 0.3

ensemble_results = model.train_resplit_ensemble(
    X_trainval=X_trainval,
    y_trainval=y_trainval,
    I=I,
    R=R,
    seeds=RUN_SEEDS,
    save_dir=ENSEMBLE_DIR,
    Market=Market,
    val_ratio=VAL_RATIO,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=1e-4,
    weight_decay=1e-4,
    device=device,
)


ensemble runs:   0%|          | 0/5 [00:00<?, ?it/s, seed=0]/home/iof_314707035/miniconda3/envs/reimage-price-trends/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.5) - (12.0)
    
  queued_call()
ensemble runs:   0%|          | 0/5 [1:00:29<?, ?it/s, seed=0]


KeyboardInterrupt: 

# Test

In [26]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

root = Path("/home/iof_314707035/Jasper")
sys.path.insert(0, str(root / "Reimage_price_trends"))

import model


In [27]:
market = "US_1993_2020"
dataset_dir = root / "Reimage_price_trends" / "data" / "training_data" / market / "I20R20S20_period_end_month"

X_test = np.load(dataset_dir / "X_test.npy")
meta_test = pd.read_csv(dataset_dir / "meta_test.csv")

for c in ["start_date", "date", "label_end_date"]:
    meta_test[c] = pd.to_datetime(meta_test[c])

print(X_test.shape)
print(meta_test.head())


(1413250, 64, 60)
   ticker start_date       date label_end_date  label       ret   I   R  \
0   10001 2000-12-21 2001-01-31     2001-03-07      0 -0.000412  20  20   
1   10001 2001-01-23 2001-02-28     2001-03-30      1  0.052281  20  20   
2   10001 2001-03-01 2001-03-30     2001-05-02      0 -0.004949  20  20   
3   10001 2001-03-29 2001-04-30     2001-05-29      1  0.076922  20  20   
4   10001 2001-05-03 2001-05-31     2001-06-29      1  0.188107  20  20   

   sample_step sample_mode sample_freq price_source ending_date  
0           20  period_end       month         crsp  2001-01-31  
1           20  period_end       month         crsp  2001-02-28  
2           20  period_end       month         crsp  2001-03-30  
3           20  period_end       month         crsp  2001-04-30  
4           20  period_end       month         crsp  2001-05-31  


In [28]:
ckpt_dir = root / "WORK_SPACE" / "new_model_res" / "D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64" / "20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn"

checkpoint_paths = [str(ckpt_dir / f"checkpoint{i}.pth.tar") for i in range(5)]

checkpoint_paths


['/home/iof_314707035/Jasper/WORK_SPACE/new_model_res/D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64/20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn/checkpoint0.pth.tar',
 '/home/iof_314707035/Jasper/WORK_SPACE/new_model_res/D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64/20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn/checkpoint1.pth.tar',
 '/home/iof_314707035/Jasper/WORK_SPACE/new_model_res/D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64/20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn/checkpoint2.pth.tar',
 '/home/iof_314707035/Jasper/WORK_SPACE/new_model_res/D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64/20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn/checkpoint3.pth.tar',
 '/home/iof_314707035/Jasper/WORK_SPACE/new_model_res/D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64/20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn/checkpoint4.pth.tar']

In [29]:
mask = (
    meta_test["ticker"].astype(str).eq("10001") &
    meta_test["start_date"].eq(pd.Timestamp("2000-12-21")) &
    meta_test["date"].eq(pd.Timestamp("2001-01-31")) &
    meta_test["label_end_date"].eq(pd.Timestamp("2001-03-07"))
)

idx = meta_test[mask].index[0]
single_image = X_test[idx:idx+1]

prob = model.average_js_cnn_ensemble_predictions(
    single_image,
    checkpoint_paths,
    year=2001,
    ws=20,
    batch_size=1,
    device="cpu",
)

print("idx:", idx)
print(meta_test.iloc[idx])
print("js_cnn ensemble prob:", float(prob[0]))


idx: 0
ticker                          10001
start_date        2000-12-21 00:00:00
date              2001-01-31 00:00:00
label_end_date    2001-03-07 00:00:00
label                               0
ret                         -0.000412
I                                  20
R                                  20
sample_step                        20
sample_mode                period_end
sample_freq                     month
price_source                     crsp
ending_date                2001-01-31
Name: 0, dtype: object
js_cnn ensemble prob: 0.039835840463638306


In [30]:
js_res_2001 = pd.read_csv(
    root / "WORK_SPACE" / "new_model_res" / "D20L3F53S31D21MP21F53S11D11MP21F53S11D11MP21C64" / "20d20p-lr1E-04-dp0.50-maTrue-vbTrue-monthlyTrained-noDelayedReturn" / "ensem_res" / "ensem5_res_2001.csv"
)

js_row = js_res_2001[js_res_2001["StockID"].astype(str) == "10001"].iloc[0]
print(js_row)


Unnamed: 0              0
StockID             10001
ending_date    2001-01-31
up_prob          0.039837
ret_val               0.0
MarketCap          2962.5
Name: 0, dtype: object


In [32]:
meta_2001 = meta_test[meta_test["date"].dt.year == 2001].copy()
idx_2001 = meta_2001.index.to_numpy()

probs_2001 = model.average_js_cnn_ensemble_predictions(
    X_test[idx_2001],
    checkpoint_paths,
    year=2001,
    ws=20,
    batch_size=128,
    device="cpu",
)

meta_2001["pred_prob_jscnn"] = probs_2001
meta_2001.head()


,ticker,start_date,date,label_end_date,label,ret,I,R,sample_step,sample_mode,sample_freq,price_source,ending_date,pred_prob_jscnn
0,10001,2000-12-21,2001-01-31,2001-03-07,0,-0.000412,20,20,20,period_end,month,crsp,2001-01-31,0.039836
1,10001,2001-01-23,2001-02-28,2001-03-30,1,0.052281,20,20,20,period_end,month,crsp,2001-02-28,0.064841
2,10001,2001-03-01,2001-03-30,2001-05-02,0,-0.004949,20,20,20,period_end,month,crsp,2001-03-30,0.050906
3,10001,2001-03-29,2001-04-30,2001-05-29,1,0.076922,20,20,20,period_end,month,crsp,2001-04-30,0.015922
4,10001,2001-05-03,2001-05-31,2001-06-29,1,0.188107,20,20,20,period_end,month,crsp,2001-05-31,0.042172


In [31]:
test_probs = model.average_js_cnn_ensemble_predictions(
    X_test,
    checkpoint_paths,
    year=2001,   # 注意：如果你要對 test set 的不同年份逐年對照，這裡要依樣本年份分開跑
    ws=20,
    batch_size=128,
    device="cpu",
)

print(test_probs.shape)
print(test_probs[:10])


KeyboardInterrupt: 

In [ ]:
js_res_2001["StockID"] = js_res_2001["StockID"].astype(str)
js_res_2001["ending_date"] = pd.to_datetime(js_res_2001["ending_date"])

compare_2001 = meta_2001.merge(
    js_res_2001,
    left_on=["ticker", "date"],
    right_on=["StockID", "ending_date"],
    how="inner",
)

compare_2001["abs_diff"] = (compare_2001["pred_prob_jscnn"] - compare_2001["up_prob"]).abs()

compare_2001[[
    "ticker", "start_date", "date", "label_end_date",
    "pred_prob_jscnn", "up_prob", "abs_diff"
]].head()


In [ ]:
print("num matched:", len(compare_2001))
print("mean abs diff:", compare_2001["abs_diff"].mean())
print("max abs diff:", compare_2001["abs_diff"].max())
